In [54]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, TargetEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [7]:
df = pd.read_csv('../data/raw/ecommerce_customer_churn_dataset.csv')

## Testing model by dropping all null columns

In [ ]:
df_dropped = df.dropna().copy()

X = df_dropped.drop('Churned', axis=1)
y = df_dropped['Churned']

Xtrain, Xrest, ytrain, yrest = train_test_split(X, y, train_size=0.7, stratify=y, random_state=42)

### check for multicollinearity


In [5]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# floats are all the numeric cols
num_cols = X.select_dtypes(exclude='object').columns
vif_data = [variance_inflation_factor(X.select_dtypes(exclude=object).values, i) for i in range(num_cols.size)]
vif = pd.DataFrame({
    'feature': num_cols,
    'VIF': vif_data
})

In [6]:
vif.sort_values('VIF')

,feature,VIF
8,Average_Order_Value,1.580026
9,Days_Since_Last_Purchase,1.997038
11,Returns_Rate,2.611092
1,Membership_Years,3.053466
14,Product_Reviews_Written,4.670723
10,Discount_Usage_Rate,4.715345
17,Payment_Method_Diversity,5.277947
19,Credit_Balance,6.133862
13,Customer_Service_Calls,6.135915
6,Wishlist_Items,6.438491


The last 2 (`Session_Duration_Avg` and `Pages_Per_Session`) appears on higher end of multicolinearity, which would correlate in an ecom. 
* More pages per session → longer session duration

* Longer session duration → more pages viewed

For now all columns are used for baseline, later will experiment with dropping and/or combining them.

### encoding categorical columns

In [7]:
Xtrain.select_dtypes('O')

,Gender,Country,City,Signup_Quarter
19443,Female,France,Toulouse,Q2
25173,Male,USA,Houston,Q3
11256,Male,USA,Phoenix,Q2
38333,Male,UK,Manchester,Q1
35963,Male,Australia,Adelaide,Q1
...,...,...,...,...
40747,Female,France,Nice,Q4
36799,Other,Germany,Cologne,Q3
41305,Female,UK,Manchester,Q2
15716,Male,Canada,Montreal,Q4


In [8]:
num_cols = Xtrain.select_dtypes(exclude='object').columns

low_cardinal_cols = ['Gender', 'Signup_Quarter', 'Country']
mid_cardinal_cols = ['City']

In [9]:
# low cardinality variables
ohe = OneHotEncoder(sparse_output=False, drop='first', dtype=np.uint8).set_output(transform='pandas')
ohe.fit(Xtrain[low_cardinal_cols])

Xtrain_lowcard_ = ohe.transform(Xtrain[low_cardinal_cols])
Xtrain = pd.concat([Xtrain, Xtrain_lowcard_], axis=1).drop(low_cardinal_cols, axis=1)

In [10]:
# moderate/mid cardinality variable

[X[col].value_counts().values for col in mid_cardinal_cols]

[array([1297, 1253, 1225, 1220, 1212,  578,  562,  526,  519,  494,  451,
         449,  426,  414,  403,  371,  366,  343,  334,  325,  315,  313,
         310,  307,  300,  299,  290,  288,  287,  271,  267,  256,  248,
         245,  238,  208,  203,  198,  189,  183])]

City acts like a subcategory with Country as main category. Performing one-hot encoding on city aswell will be a dimensionality curse and multicolinearity issue.

From eda, these categorical variables show little-to-no relation with target variable (`Churned`). 

In [11]:
# Frequency encoding for City
freq = Xtrain['City'].value_counts(normalize=True)

Xtrain['City_FE'] = Xtrain['City'].map(freq)

In [12]:
Xtrain = Xtrain.drop(columns='City')

In [13]:
Xtrain.dtypes.value_counts()

float64    21
uint8      12
Name: count, dtype: int64

All features are numeric, can feed to ml model

In [14]:
lr = LogisticRegression()
lr.fit(Xtrain, ytrain)

LogisticRegression()

In [ ]:
# one hot encoding - categorical variables
Xrest_lowcard_ = ohe.transform(Xrest[low_cardinal_cols])
Xrest = pd.concat([Xrest, Xrest_lowcard_], axis=1).drop(low_cardinal_cols, axis=1)

In [17]:
# frequency encoding - cateogrical variables
Xrest['City_FE'] = Xrest['City'].map(freq)
Xrest = Xrest.drop(columns='City')

In [18]:
lr.score(Xrest, yrest)

0.7690454124189064

## Testing model with missing data imputation

In [3]:
from sklearn.impute import SimpleImputer

In [21]:
missing_flag = df.isna().any(axis=1).astype(int)

In [55]:
X = df.drop('Churned', axis=1)
y = df['Churned']

In [106]:
Xtrain, Xrest, ytrain, yrest = train_test_split(X, y, train_size=0.7, stratify=np.array(missing_flag, y))

In [57]:
mean_imputer = SimpleImputer().set_output(transform='pandas')

In [58]:
mean_imputer.fit(Xtrain.select_dtypes(exclude=object))

SimpleImputer()

In [ ]:
Xtrain_missing_flag = Xtrain.isna().any()
Xtrain_missing_fix = mean_imputer.transform(Xtrain.select_dtypes(exclude=object))
Xtrain = Xtrain.combine_first(Xtrain_missing_fix)

In [64]:
# categorical 
num_cols = Xtrain.select_dtypes(exclude='object').columns

low_cardinal_cols = ['Gender', 'Signup_Quarter', 'Country']
mid_cardinal_cols = ['City']

### low cardinal

In [82]:
ohe = OneHotEncoder(sparse_output=False, drop='first', dtype=np.uint8, handle_unknown='error').set_output(transform='pandas')
ohe.fit(Xtrain[low_cardinal_cols])

OneHotEncoder(drop='first', dtype=<class 'numpy.uint8'>, sparse_output=False)

In [84]:
Xtrain_lowcard_ = ohe.transform(Xtrain[low_cardinal_cols])
Xtrain = pd.concat([Xtrain, Xtrain_lowcard_], axis=1).drop(low_cardinal_cols, axis=1)

### Medium/Moderate cardinal

In [94]:
freq_map = Xtrain[mid_cardinal_cols].value_counts(normalize=True)
Xtrain[mid_cardinal_cols] = Xtrain[mid_cardinal_cols].replace(freq_map)

In [95]:
Xtrain.dtypes.value_counts()

float64    21
uint8      12
Name: count, dtype: int64

### train model

In [96]:
lr = LogisticRegression()
lr.fit(Xtrain, ytrain)

LogisticRegression()

### evaluate model

In [107]:
# impute missing values
Xrest_missing_flag = Xrest.isna().any(axis=1)
Xrest_missing_fix = mean_imputer.transform(Xrest.select_dtypes(exclude=object))
Xrest = Xrest.combine_first(Xrest_missing_fix)

In [108]:
# low cardinal features
Xrest_lowcard_ = ohe.transform(Xrest[low_cardinal_cols])
Xrest = pd.concat([Xrest, Xrest_lowcard_], axis=1).drop(columns=low_cardinal_cols)

In [109]:
# mid cardinal features
Xrest[mid_cardinal_cols] = Xrest[mid_cardinal_cols].replace(freq_map)

In [110]:
lr.score(Xrest, yrest)

0.7695333333333333

In [113]:
lr.score(Xrest[~Xrest_missing_flag], yrest[~Xrest_missing_flag])

0.7775718257645968